# Módulo 04 · Aula 5 — SQL e pandas juntos

**Capacitação Introdutória de Ciência de Dados · FEA.dev**

---

Você aprendeu duas ferramentas que fazem coisas parecidas. Esta aula responde à pergunta
prática que sobra: **onde termina uma e começa a outra?**

E trata de três assuntos que separam quem escreve SQL de quem escreve SQL em produção:
consultas com parâmetros (e por que nunca montar SQL com `f-string`), limpeza de dados no
banco, e o que muda quando você troca de SQLite para PostgreSQL ou BigQuery.

Ao final desta aula você vai:

- passar parâmetros para uma consulta com segurança;
- decidir o que fazer no banco e o que fazer no pandas;
- percorrer uma análise completa de ponta a ponta: pergunta → SQL → pandas → gráfico;
- escrever um resultado de volta no banco com `to_sql`;
- saber o que esperar ao mudar de banco.

**Tempo estimado:** 60 minutos.

### Antes de começar — se você está no Google Colab

Este notebook lê o banco de dados da pasta `data/` do repositório, e no Colab a máquina começa vazia. **Execute a célula abaixo antes de qualquer outra**: ela traz o repositório e entra na pasta deste módulo, de modo que os caminhos `../data/...` usados no material funcionem sem alteração.

No VS Code ou no Jupyter local a célula não faz nada — os arquivos já estão no seu disco.

In [ ]:
# Setup do Google Colab.
# Traz o repositório da capacitação e entra na pasta deste módulo, para que os
# caminhos "../data/..." usados no material funcionem sem nenhuma alteração.
# Fora do Colab (VS Code, Jupyter local) esta célula não faz nada.
# Pode ser executada mais de uma vez sem problema.
import os
import subprocess
import sys

PASTA_DESTE_MODULO = "04_SQL"
REPOSITORIO = "https://github.com/gustavokatsuo/Introducao-a-Ciencia-de-Dados.git"

if "google.colab" in sys.modules and not os.path.isdir("../data"):
    destino = "/content/Introducao-a-Ciencia-de-Dados"
    if not os.path.isdir(destino):
        print("Baixando o material da capacitação...")
        subprocess.run(["git", "clone", "--depth", "1", REPOSITORIO, destino], check=True)
    os.chdir(os.path.join(destino, PASTA_DESTE_MODULO))
    print("Pronto. Pasta de trabalho:", os.getcwd())

In [ ]:
import sqlite3

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

%matplotlib inline
sns.set_theme(style="whitegrid", palette="deep")
pd.set_option("display.max_columns", 25)
pd.set_option("display.width", 140)

conexao = sqlite3.connect("../data/capacitacao.db")


def consultar(sql, parametros=None):
    return pd.read_sql_query(sql, conexao, params=parametros)


print("Conectado.")

## 1. Consultas com parâmetros

Toda consulta útil acaba precisando variar: o ticker, a data de corte, o setor. A tentação
é montar a string com `f-string`.

**Não faça isso.** Nunca.

```python
# ERRADO — não escreva assim, nem em exercício
ticker = "PETR4"
consultar(f"SELECT * FROM cotacoes WHERE ticker = '{ticker}'")
```

O jeito certo é deixar um marcador `?` na consulta e passar os valores separadamente. O
driver do banco cuida de encaixá-los.

In [ ]:
consultar(
    """
    SELECT ticker, data, ROUND(fechamento_ajustado, 2) AS fechamento
    FROM cotacoes
    WHERE ticker = ?
      AND data >= ?
    ORDER BY data DESC
    LIMIT 5
    """,
    parametros=("VALE3", "2025-12-01"),
)

### Por que isso importa

São dois motivos, e o segundo é o que dá manchete.

**1. Correção.** Um nome com apóstrofo — `O'Brien`, `Sant'Ana` — quebra a string montada à
mão, porque a aspa fecha o texto no meio. Com parâmetro, funciona sem você pensar nisso.

In [ ]:
# Com f-string, a aspa em O'Brien encerraria o texto e a consulta viraria sintaxe inválida.
# Com parâmetro, é apenas um valor:
consultar("SELECT COUNT(*) AS achou FROM clientes WHERE nome = ?", parametros=("O'Brien",))

**2. Segurança: injeção de SQL.** Se o valor vier de fora — um formulário, um
arquivo, uma API — quem escreve o valor passa a escrever *parte da sua consulta*. Um valor
cuidadosamente escolhido pode transformar um `SELECT` inofensivo em um `DROP TABLE`.

Isso é, há duas décadas, uma das falhas de segurança mais exploradas do mundo. E a defesa
é literalmente trocar `f"...{x}..."` por `?` — sem custo de desempenho, sem esforço.

Vamos ver o mecanismo, sem estragar nada: usamos uma cópia descartável do banco, na
memória.

In [ ]:
# Um banco de brinquedo, em memória, só para demonstrar o mecanismo.
brinquedo = sqlite3.connect(":memory:")
brinquedo.execute("CREATE TABLE segredo (valor TEXT)")
brinquedo.execute("INSERT INTO segredo VALUES ('dado importante')")
brinquedo.commit()

# Um valor "vindo do usuário" que fecha a string e emenda outra condição:
entrada_maliciosa = "x' OR '1'='1"

consulta_montada = f"SELECT * FROM segredo WHERE valor = '{entrada_maliciosa}'"
print("A consulta que o banco vai receber:")
print("   ", consulta_montada)
print()
print("Resultado — devolveu a linha, apesar de 'x' não existir:")
print("   ", brinquedo.execute(consulta_montada).fetchall())

In [ ]:
# A mesma entrada, com parâmetro: tratada como TEXTO, não como código.
print("Com parâmetro:",
      brinquedo.execute("SELECT * FROM segredo WHERE valor = ?", (entrada_maliciosa,)).fetchall())
print("Vazio, como deveria ser — não existe cliente com esse nome literal.")

brinquedo.close()

> **A regra, sem exceção:** valores vão em `?`; só o *texto fixo* da consulta é
> escrito por você. Se você se pegar montando SQL com `f-string`, pare.
>
> Um detalhe: `?` serve para **valores**, não para nomes de tabela ou coluna. Se precisar
> variar um nome de coluna, valide contra uma lista de nomes permitidos — nunca interpole
> o que veio de fora.

## 2. Onde traçar a linha

A pergunta prática do dia a dia. A resposta curta:

> **Reduza no banco. Analise no pandas.**

O banco é imbatível para filtrar, juntar e agregar sobre muitas linhas — foi construído
para isso, e o que ele devolve já vem menor. O pandas é imbatível para o que vem depois:
explorar, visualizar, modelar, iterar.

| Faça no **banco** | Faça no **pandas** |
|---|---|
| filtrar linhas e colunas | gráficos |
| juntar tabelas | estatística descritiva e testes |
| agregar (`GROUP BY`) | transformações exploratórias, indo e voltando |
| janelas sobre a série completa | qualquer coisa que exija iterar rápido |
| tudo que **reduz volume** | tudo que **precisa de olho humano** |

**O antipadrão mais comum** é `SELECT * FROM tabela` seguido de filtragem em pandas. Numa
tabela de 200 milhões de linhas, isso transfere tudo pela rede para jogar 99% fora. O
filtro custa quase nada se for feito no banco.

**O antipadrão oposto** também existe: tentar fazer regressão, gráfico ou análise
exploratória em SQL. Dá para torturar o SQL até ele fazer, mas é lento de escrever e
ilegível de manter.

## 3. Uma análise de ponta a ponta

Vamos fechar o módulo com o ciclo completo, no formato do módulo 03: pergunta, exploração,
descoberta.

> **Pergunta:** o setor Financeiro teve comportamento mais estável que os demais em 2025?

O plano: o banco reduz cinco anos de pregões diários a uma tabela de retornos mensais por
setor — algumas dezenas de linhas —, e o pandas cuida da análise e do gráfico.

In [ ]:
retornos = consultar("""
    WITH com_retorno AS (
        SELECT
            c.ticker,
            c.data,
            e.setor,
            c.fechamento_ajustado,
            LAG(c.fechamento_ajustado) OVER (
                PARTITION BY c.ticker ORDER BY c.data
            ) AS anterior
        FROM cotacoes c
        INNER JOIN empresas e ON c.ticker = e.ticker
    )
    SELECT
        setor,
        ticker,
        strftime('%Y-%m', data) AS ano_mes,
        (fechamento_ajustado / anterior - 1) * 100 AS retorno_diario_pct
    FROM com_retorno
    WHERE anterior IS NOT NULL
      AND data >= ?
""", parametros=("2025-01-01",))

print("Linhas trazidas do banco:", len(retornos))
retornos.head()

Repare no que **não** aconteceu: não trouxemos 9.968 linhas. O `JOIN`, a janela e
o filtro rodaram no banco, e chegou só o necessário.

Agora o pandas assume.

In [ ]:
resumo = (
    retornos.groupby("setor")["retorno_diario_pct"]
    .agg(observacoes="count", retorno_medio="mean", volatilidade="std")
    .round(3)
    .sort_values("volatilidade")
)
resumo

In [ ]:
fig, eixos = plt.subplots(1, 2, figsize=(13, 4.5))

ordem = resumo.index.tolist()

sns.boxplot(data=retornos, x="retorno_diario_pct", y="setor", order=ordem,
            ax=eixos[0], showfliers=False)
eixos[0].set_title("Distribuição dos retornos diários por setor — 2025")
eixos[0].set_xlabel("retorno diário (%)")
eixos[0].set_ylabel("")
eixos[0].axvline(0, color="#595959", linewidth=1, linestyle="--")

eixos[1].barh(resumo.index, resumo["volatilidade"], color="#1f4e79")
eixos[1].set_title("Volatilidade (desvio padrão do retorno diário)")
eixos[1].set_xlabel("desvio padrão (p.p.)")
eixos[1].invert_yaxis()

fig.tight_layout()
plt.show()

### Descoberta

O Financeiro **não** é o setor mais estável — e a leitura do gráfico exige o cuidado que o
módulo 03 insistiu em treinar.

Olhe a coluna `observacoes` do resumo antes de concluir qualquer coisa: os setores têm
número muito diferente de observações, porque a nossa amostra tem 3 papéis no Financeiro e
**um só** em vários outros. Um "setor" representado por uma única empresa não mede o setor;
mede aquela empresa.

Ou seja: a pergunta original não é respondível com esta base, e a resposta honesta é dizer
isso, não escolher o número que parece bom. É exatamente o ciclo 2 do notebook de EDA do
módulo 03 — descoberta que depende de um único ponto é artefato.

> O SQL não protege você disso. Ele responde exatamente o que foi perguntado, muito rápido,
> mesmo quando a pergunta não faz sentido para os dados que existem. O julgamento continua
> sendo seu.

In [ ]:
# A conferência que deveria vir ANTES do gráfico
consultar("""
    SELECT
        e.setor,
        COUNT(DISTINCT e.ticker) AS papeis,
        GROUP_CONCAT(DISTINCT e.ticker) AS quais
    FROM empresas e
    GROUP BY e.setor
    ORDER BY papeis DESC
""")

## 4. Escrevendo de volta: `to_sql`

Análise que vale a pena costuma virar tabela. `DataFrame.to_sql` grava um DataFrame como
tabela do banco.

In [ ]:
resumo.reset_index().to_sql("resumo_setor_2025", conexao,
                             if_exists="replace", index=False)

consultar("SELECT * FROM resumo_setor_2025 ORDER BY volatilidade")

O `if_exists` decide o que fazer se a tabela já existir: `"fail"` (o padrão,
levanta erro), `"replace"` (apaga e recria) ou `"append"` (acrescenta linhas).

> **`"replace"` apaga a tabela inteira.** Em um banco compartilhado, isso destrói o
> trabalho de outra pessoa sem perguntar. Em ambiente profissional, escrita quase sempre
> exige permissão específica — e é comum que analistas tenham acesso só de leitura,
> justamente por isso.

In [ ]:
# Limpando a tabela de exemplo, para o banco do repositório não mudar
conexao.execute("DROP TABLE IF EXISTS resumo_setor_2025")
conexao.commit()
print("Tabela de exemplo removida.")

## 5. Limpeza de dados em SQL

O módulo 02 limpou a base de clientes com pandas. Boa parte daquilo tem versão em SQL — e
quando a base é grande, limpar no banco é a única opção viável.

Os mesmos defeitos, os mesmos remédios:

In [ ]:
consultar("""
    SELECT
        -- categorias inconsistentes: TRIM tira espaços, UPPER uniformiza a caixa
        UPPER(TRIM(perfil_investidor)) AS perfil,
        COUNT(*) AS clientes
    FROM clientes
    WHERE perfil_investidor IS NOT NULL
    GROUP BY perfil
    ORDER BY clientes DESC
""")

Doze grafias viraram três categorias — o mesmo resultado do
`.str.strip().str.upper()` do módulo 02.

Agora o caso mais interessante: `patrimonio_investido` é texto em formato brasileiro. Antes
de escrever qualquer conversão, **olhe quantos formatos existem** — essa é a etapa que
costuma ser pulada.

In [ ]:
consultar("""
    SELECT
        CASE
            WHEN patrimonio_investido IS NULL      THEN 'vazio'
            WHEN patrimonio_investido LIKE 'R$%'   THEN 'com prefixo R$'
            ELSE                                        'sem prefixo'
        END AS formato,
        COUNT(*) AS linhas,
        MIN(patrimonio_investido) AS exemplo
    FROM clientes
    GROUP BY formato
""")

**Dois formatos, não um.** 91 valores trazem `'R$ '` na frente; 293 não. Os dois
usam ponto como separador de milhar e vírgula como decimal.

Essa contagem muda tudo. A tentação é escrever um `CASE` que converte só quem tem `R$` e
manda o resto direto para o `CAST`:

```sql
WHEN patrimonio_investido LIKE 'R$%' THEN CAST(REPLACE(...) AS REAL)
ELSE CAST(patrimonio_investido AS REAL)     -- ERRADO
```

O `ELSE` aí destrói 293 valores. `CAST('412.666,21' AS REAL)` no SQLite não dá erro: ele lê
até o primeiro caractere inválido e devolve **412.666**. Um patrimônio de quatrocentos mil
vira quatrocentos, a média despenca, e nada acusa.

A correção é não ter `ELSE` especial. Os três `REPLACE` valem para todo mundo — em um valor
sem `'R$ '`, o primeiro simplesmente não encontra nada e não faz nada:

In [ ]:
consultar("""
    WITH limpo AS (
        SELECT
            patrimonio_investido AS original,
            CASE
                WHEN patrimonio_investido IS NULL THEN NULL
                ELSE CAST(
                    REPLACE(REPLACE(REPLACE(patrimonio_investido, 'R$ ', ''), '.', ''), ',', '.')
                    AS REAL)
            END AS patrimonio
        FROM clientes
    )
    SELECT original, ROUND(patrimonio, 2) AS convertido
    FROM limpo
    WHERE original IS NOT NULL
    ORDER BY patrimonio DESC
    LIMIT 6
""")

Os dois formatos convertidos pela mesma expressão, sem ramo especial.

A ordem dos `REPLACE` importa: primeiro some o `'R$ '`, depois o ponto de
milhar, e só então a vírgula vira ponto. Inverter os dois últimos passos transformaria
`412.666,21` em `412.666.21`, que o `CAST` leria como `412` — silenciosamente, sem erro.

> **`CAST` no SQLite nunca reclama.** `CAST('abacaxi' AS REAL)` devolve `0.0`; texto
> parcialmente numérico vira o pedaço que deu para ler, como no `412.666,21` acima. Em
> Postgres, os dois casos levantam exceção — e uma exceção é infinitamente melhor que um
> número errado, porque você fica sabendo.

Vale a comparação honesta: essa limpeza em pandas é uma linha
(`.str.replace(...).astype(float)`), e aqui são oito. **Nem toda limpeza compensa em SQL.**
A regra prática: se a base cabe na memória, limpe em pandas; se não cabe, limpe no banco e
aceite a verbosidade.

## 6. Quando você mudar de banco

Tudo que você escreveu neste módulo funciona em qualquer banco relacional — `SELECT`,
`WHERE`, `GROUP BY`, `HAVING`, `JOIN`, CTE, funções de janela são padrão SQL.

O que muda:

| Assunto | SQLite | PostgreSQL | BigQuery |
|---|---|---|---|
| extrair ano | `strftime('%Y', d)` | `EXTRACT(YEAR FROM d)` | `EXTRACT(YEAR FROM d)` |
| truncar mês | `strftime('%Y-%m', d)` | `DATE_TRUNC('month', d)` | `DATE_TRUNC(d, MONTH)` |
| concatenar | `||` | `||` | `CONCAT()` |
| tipo de data | não existe: texto | `DATE`, `TIMESTAMP` | `DATE`, `TIMESTAMP` |
| limite de linhas | `LIMIT n` | `LIMIT n` | `LIMIT n` |
| tipos | flexíveis, quase sugestões | estritos | estritos |

E as três diferenças de **comportamento** que este módulo mostrou, todas do mesmo tipo — o
SQLite deixa passar o que os outros recusam:

1. **apelido do `SELECT` usado no `WHERE`** (aula 1) — erro no Postgres;
2. **coluna fora do `GROUP BY` e fora de agregação** (aula 2) — erro no Postgres; no SQLite,
   devolve valor arbitrário;
3. **`CAST` de texto inconversível** (esta aula) — erro no Postgres; no SQLite, vira zero.

O padrão é claro: **o SQLite perdoa, e o perdão vira bug quando você migra.** Escreva como
se ele fosse estrito, e a mudança de banco será um detalhe.

## 7. O módulo inteiro em uma tabela

| Preciso... | pandas | SQL |
|---|---|---|
| escolher colunas | `df[["a", "b"]]` | `SELECT a, b` |
| filtrar linhas | máscara booleana | `WHERE` |
| ordenar | `.sort_values()` | `ORDER BY` |
| primeiras n | `.head(n)` | `LIMIT n` |
| valores únicos | `.unique()` | `DISTINCT` |
| contar | `len(df)`, `.count()` | `COUNT(*)`, `COUNT(col)` |
| agrupar e resumir | `.groupby().agg()` | `GROUP BY` + agregações |
| filtrar grupos | `.filter()` | `HAVING` |
| classificar em faixas | `np.select`, `pd.cut` | `CASE WHEN` |
| tabela cruzada | `.pivot_table()` | `SUM(CASE WHEN ...)` |
| juntar tabelas | `.merge()` | `JOIN ... ON` |
| conferir a junção | `validate=`, `indicator=` | contar antes e depois |
| etapas nomeadas | variáveis intermediárias | `WITH ... AS` |
| linha anterior | `.shift()` | `LAG() OVER` |
| variação % | `.pct_change()` | `LAG` + divisão |
| média móvel | `.rolling(n).mean()` | `AVG() OVER (ROWS BETWEEN ...)` |
| acumulado | `.cumsum()`, `.cummax()` | `SUM()/MAX() OVER (UNBOUNDED PRECEDING)` |
| ranking por grupo | `.groupby().rank()` | `RANK() OVER (PARTITION BY ...)` |
| agregado sem colapsar | `.transform()` | `AVG() OVER (PARTITION BY ...)` |
| ler | `pd.read_csv` | `pd.read_sql_query` |
| escrever | `.to_csv()` | `.to_sql()` |

## 8. Recapitulando

- **Valores vão em `?`, nunca em `f-string`.** É correção (apóstrofos) e é segurança
  (injeção de SQL). `?` vale para valores, não para nomes de tabela ou coluna.
- **Reduza no banco, analise no pandas.** Filtrar, juntar e agregar do lado do banco;
  gráfico, estatística e exploração do lado do pandas.
- `SELECT *` seguido de filtro em pandas é o antipadrão mais comum. Tentar fazer análise
  exploratória em SQL é o oposto.
- `to_sql` escreve de volta; `if_exists="replace"` **apaga a tabela** — cuidado em banco
  compartilhado.
- Limpeza em SQL é possível e às vezes obrigatória, mas costuma custar oito linhas onde o
  pandas gasta uma. Escolha pela dimensão da base.
- A ordem dos `REPLACE` importa; `CAST` inconversível vira `0.0` no SQLite, em silêncio.
- **O SQLite perdoa o que os outros bancos recusam.** Escreva como se ele fosse estrito.
- Nada disso substitui julgamento: o banco responde rápido e exatamente o que foi
  perguntado, inclusive quando a pergunta não faz sentido para os dados.

---

**Fim do módulo 04.** A lista de exercícios correspondente é
[`lista_04_sql.ipynb`](../05_Exercicios/lista_04_sql.ipynb), com gabarito em arquivo
separado.

In [ ]:
conexao.close()
print("Conexão fechada.")